# Change pixel colors based on pose estimation

## Ideas
- get lines, expand and blur it
- change pixel color according to their distance from pose landmarks

## Import modules

In [1]:
# Import internal modules
# import math
from pathlib import Path
# import random
from typing import Dict, List, Optional, Set, Tuple, TypedDict

# Import 3rd party modules
import cv2
import numpy as np
from scipy.interpolate import interp1d
# import tensorflow as tf
# import matplotlib.pyplot as plt
# from matplotlib import colors
from IPython import display
from numba import njit

# Import local modules
from pose_estimation.play_with_particles.particle import Particle
from pose_estimation.play_with_particles.environment import Environment
# from core.pose_estimation.play_with_particles.njit_move import njit_move_particles
from core.pose_estimation.play_with_particles.utils import draw_mediapipe_connections

# from core.utils.renderer.get_resize_interpolation import get_interpolation
from core.utils.renderer.resizer import resize_with_crop
from core.utils.project_manager import Project

## Set up project

In [2]:
# create project
project = Project(project_dir="assets/images/pose_estimation")

## Define constants & variables

In [3]:
NB_PARTICLES = 250
PARTICLE_MIN_SIZE: int = 10 # 2
PARTICLE_MAX_SIZE: int = 20 # 10
PARTICLE_MASS = 50
THICKNESS = -1

LINE_PARTICLE_SIZE_PX = 50

# confidence threshold for pose estimation
CONFIDENCE_THRESHOLD = 0.5

# set text parameters
FONT = cv2.FONT_HERSHEY_SIMPLEX
FONT_SCALE = 0.7
TEXT_THICKNESS = 1

# set colors
BLACK = (0,0,0)
BLUE = (255, 178, 50)
YELLOW = (0,255, 255)

## load pose landmarks npy file

In [3]:
# load mediapipe pose landmarks
landmarks = np.load("assets/images/gagu/landmarks.npy")
landmarks.shape

(1442, 23, 2, 2)

## render blurred lines

In [10]:
caption: str = f"draw_blurred_line_{NB_PARTICLES}"
NB_PARTICLES = 10000
PARTICLE_MIN_SIZE: int = 2 # 2
PARTICLE_MAX_SIZE: int = 10 # 10
PARTICLE_MASS = 50
THICKNESS = -1

LINE_PARTICLE_SIZE_PX = 75


# set input & output video path
video_path = Path("assets/images/gagu/gagu_original.mp4")
out_path = project.project_dir / f"{video_path.stem}_{caption}.mp4"
out_np_path = project.project_dir / f"{video_path.stem}_{caption}.npy"

# Initialize video stream
video_cap = cv2.VideoCapture(str(video_path))

# get video parameters
video_nb_frames = int(video_cap.get(cv2.CAP_PROP_FRAME_COUNT))
video_fps = video_cap.get(cv2.CAP_PROP_FPS)
video_width = int(video_cap.get(cv2.CAP_PROP_FRAME_WIDTH))
video_height = int(video_cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f"number of frames = {video_nb_frames}")
print(f"fps = {video_fps}")
print(f"video width = {video_width}")
print(f"video height = {video_height}")

# set codec for output video
codec = "H264"

rotate = False
resize = False

# set output shape
# out_height, out_width, out_channel = 1920, 1080, 3
out_height, out_width, out_channel = video_height, video_width, 3

# create a videoWriter object
fourcc = cv2.VideoWriter_fourcc(*codec)
out_video = cv2.VideoWriter(filename=str(out_path), fourcc=fourcc, fps=video_fps, frameSize=(out_width, out_height))

frame_nb = 0

while True:

    # read video stream
    ret, frame = video_cap.read()

    blurred = np.zeros_like(frame)

    # break out of loop if empty frame
    if not ret:
        print(f"frame is empty")
        break
    
    # 4. render predictions
    lines = landmarks[frame_nb]
    _ = draw_mediapipe_connections(blurred, lines, LINE_PARTICLE_SIZE_PX, PARTICLE_MASS, thickness=80, draw=True, color='b')

    blurred = cv2.GaussianBlur(blurred, (101, 101), 0)

    frame = cv2.addWeighted(frame, 1, blurred, 0.8, 0)

    # rotate & resize frame if asked
    if rotate:
        frame = np.rot90(frame, -1)
    
    if resize:
        frame = resize_with_crop(frame, ref_img_shape=(out_height, out_width, out_channel))

    cv2.imshow(caption, frame)
    
    # wait for a key 
    # 0xFF to check what key we pressed on the keyboard
    key = cv2.waitKey(10) & 0xFF

    # break out of the stream loop if esc is pressed
    if key == 27 or key == ord('q'):        
        break

    # write output frame
    out_video.write(frame)

    print(frame_nb)

    # clear previous output when new fake images are displayed
    display.clear_output(wait=True)

    frame_nb += 1

# release video stream & video rendering
video_cap.release()
out_video.release()

# quit windows
cv2.destroyAllWindows()
cv2.waitKey(1) # workaround to effectively close window on mac

-1

## change pixel color according to their distance from pose landmarks

In [20]:
@njit
def get_best_distances_grid_landmarks(grid_positions, line_particle_yxs):

    best_distances = []
    # get closest pixel-line_particle
    for grid_yx in grid_positions:

        best_dist = np.inf
        
        grid_y, grid_x = grid_yx

        for line_particle_yx in line_particle_yxs:

            line_particle_y, line_particle_x = line_particle_yx
            
            dist = np.sqrt((grid_x-line_particle_x)**2 + (grid_y-line_particle_y)**2)

            if dist < best_dist:
                best_dist = dist

        best_distances.append(best_dist)

    return best_distances

In [87]:
NB_PARTICLES = 10000
PARTICLE_MIN_SIZE: int = 2 # 2
PARTICLE_MAX_SIZE: int = 10 # 10
PARTICLE_MASS = 50
THICKNESS = -1

LINE_PARTICLE_SIZE_PX = 75

caption: str = f"change_pixel_color_hue"

# set input & output video path
video_path = Path("assets/images/gagu/gagu_original.mp4")
out_path = project.project_dir / f"{video_path.stem}_{caption}.mp4"
out_np_path = project.project_dir / f"{video_path.stem}_{caption}.npy"

# Initialize video stream
video_cap = cv2.VideoCapture(str(video_path))

# get video parameters
video_nb_frames = int(video_cap.get(cv2.CAP_PROP_FRAME_COUNT))
video_fps = video_cap.get(cv2.CAP_PROP_FPS)
video_width = int(video_cap.get(cv2.CAP_PROP_FRAME_WIDTH))
video_height = int(video_cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f"number of frames = {video_nb_frames}")
print(f"fps = {video_fps}")
print(f"video width = {video_width}")
print(f"video height = {video_height}")

# set codec for output video
codec = "H264"

rotate = False
resize = False

# set output shape
# out_height, out_width, out_channel = 1920, 1080, 3
out_height, out_width, out_channel = video_height, video_width, 3

# create a videoWriter object
fourcc = cv2.VideoWriter_fourcc(*codec)
out_video = cv2.VideoWriter(filename=str(out_path), fourcc=fourcc, fps=video_fps, frameSize=(out_width, out_height))

NB_COLS = video_width
NB_ROWS = video_height

grid_positions = [(grid_y, grid_x) for grid_x in range(NB_COLS) for grid_y in range(NB_ROWS)]

# get range of integers
# hue_range = np.arange(0, 179, dtype=np.uint64)
max_dist = np.max((video_width, video_height))//2

frame_nb = 0

while True:

    # read video stream
    ret, frame = video_cap.read()

    # blurred = np.zeros_like(frame)

    # break out of loop if empty frame
    if not ret:
        print(f"frame is empty")
        break
    
    # 4. render predictions
    lines = landmarks[frame_nb]
    line_particles = draw_mediapipe_connections(frame, lines, LINE_PARTICLE_SIZE_PX, PARTICLE_MASS, thickness=80, draw=False, color='b')

    # convert frame to hsv
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    frame[:,:,1] = 255

    # get closest pixel-line_particle
    # for grid_yx in grid_positions:

    #     best_dist = np.inf
        
    #     grid_y, grid_x = grid_yx

    #     grid_yx = np.array(grid_yx)

    #     for line_particle in line_particles:
    #         line_particle_yx = np.array((line_particle.y_px, line_particle.x_px), dtype=np.int)
            
    #         dist = np.linalg.norm(np.array(grid_yx) - line_particle_yx)

    #         if dist < best_dist:
    #             best_dist = dist

    #     # change pixel color value according to best dist
    #     pixel_value = frame[grid_y, grid_x, 2]
    #     frame[grid_y, grid_x, 2] = pixel_value//(best_dist + 0.01)

    line_particle_yxs = [(int(line_particle.y_px), int(line_particle.x_px)) for line_particle in line_particles]

    best_distances = get_best_distances_grid_landmarks(grid_positions, line_particle_yxs)

    for i, grid_yx in enumerate(grid_positions):

        grid_y, grid_x = grid_yx
        best_dist = int(best_distances[i])
        # max_dist = int(np.max(best_distances))
        # color_grads = np.linspace(0,179, max_dist)
        # get y range with same number of data points as in x range
        # dist_range = np.linspace(0, max_dist, len(hue_range), dtype=np.uint64)

        # # create interpolate object
        # interp1d_fct = interp1d(dist_range, hue_range, kind = 'linear')

        # # get interpolated data
        # color_grad = interp1d_fct(best_dist)
        

        # change pixel color value according to best dist
        # pixel_value = frame[grid_y, grid_x, 0]
        # try:
        #     frame[grid_y, grid_x, 0] = color_grad # color_grads[best_dist]
        # except:
        #     print(f"fail for {grid_y}, {grid_x}, {best_dist} {pixel_value}")
        # frame[grid_y, grid_x, 0] = color_grad # color_grads[best_dist]
        frame[grid_y, grid_x, 0] = int(179 - 179*best_dist/max_dist)
        # frame[grid_y, grid_x, 2] = int(255 - (255*best_dist/max_dist))
        # frame[grid_y, grid_x, 0] = 179

    # convert frame back to bgr
    frame = cv2.cvtColor(frame, cv2.COLOR_HSV2BGR)

    # blurred = cv2.GaussianBlur(blurred, (101, 101), 0)

    # frame = cv2.addWeighted(frame, 1, blurred, 0.8, 0)

    # rotate & resize frame if asked
    if rotate:
        frame = np.rot90(frame, -1)
    
    if resize:
        frame = resize_with_crop(frame, ref_img_shape=(out_height, out_width, out_channel))

    cv2.imshow(caption, frame)
    
    # wait for a key 
    # 0xFF to check what key we pressed on the keyboard
    key = cv2.waitKey(10) & 0xFF

    # break out of the stream loop if esc is pressed
    if key == 27 or key == ord('q'):        
        break

    # write output frame
    out_video.write(frame)

    print(frame_nb)

    # clear previous output when new fake images are displayed
    display.clear_output(wait=True)

    frame_nb += 1

# release video stream & video rendering
video_cap.release()
out_video.release()

# quit windows
cv2.destroyAllWindows()
cv2.waitKey(1) # workaround to effectively close window on mac

-1

: 

In [86]:
# release video stream & video rendering
video_cap.release()
out_video.release()

# quit windows
cv2.destroyAllWindows()
cv2.waitKey(1) # workaround to effectively close window on mac

-1

In [34]:
255 - (500.81302369353693/255)

253.03602735806456

In [76]:
int(179*best_dist/max_dist)

110